# Python for Data Engineers – Platinum Notebook

**Layer:** Platinum (built from `gold.md` – curated, corrected and deduplicated handwritten notes, 19 pages)
**Updated:** 2026-11-08 · **Target:** Databricks Runtime 17+ (Apache Spark 4.x, Python 3.12+) – classic or serverless compute

Every chapter has:
* 📘 **Markdown cells** – definitions (corrected versions of the handwritten notes)
* 🐍 **Python cells** – pure-Python samples
* ⚡ **PySpark cells** – the same idea applied to Spark DataFrames
* 🆕 **What's new** – information that was *not* in the original notes (Python 3.10 – 3.15, Spark 4.x, Databricks)

| # | Chapter |
|---|---|
| 0 | Environment check |
| 1 | Variables, data types & casting |
| 2 | Strings |
| 3 | Operators & control flow |
| 4 | Functions, scope, lambdas & decorators |
| 5 | Collections & built-in functions |
| 6 | Comprehensions & functional tools |
| 7 | Generators & iterators |
| 8 | Object-oriented programming & dataclasses |
| 9 | Modules, packages & imports |
| 10 | Type hints & tooling |
| 11 | Files & JSON |
| 12 | Errors & exceptions |
| 13 | Async IO |
| 14 | Pandas API on Spark |
| 15 | What's new in Spark 4.x & Databricks |
| 16 | Capstone – Medallion pipeline in PySpark |

> **Import:** Workspace → *Import* → *File* → select `platinum.ipynb` (Jupyter Notebook format, nbformat 4).

## 0. Environment check
`spark` (SparkSession) and `display()` are pre-defined in every Databricks notebook.
On **serverless** and **standard (shared) access mode** compute the notebook talks to Spark through **Spark Connect** – `spark.sparkContext` and RDD APIs are not available, so this notebook uses only the DataFrame API.

In [ ]:
import sys
import platform

print(f"Python  : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"Spark   : {spark.version}")
print(f"Connect : {'connect' in type(spark).__module__}")  # True on serverless / Spark Connect

PY_314: bool = sys.version_info >= (3, 14)
PY_315: bool = sys.version_info >= (3, 15)

## 1. Variables, data types & casting

* A **variable** is a name bound to an object; it is created on assignment.
* Python is **dynamically** (a name can be re-bound to another type) **and strongly** typed (`5 + "a"` → `TypeError`).
* **Naming rules:** start with a letter or `_`, case-sensitive, no keywords. **Never shadow built-ins** (`list`, `str`, `dict`, `id`, …) – the handwritten note `str = "Hello"` breaks `str(3)` afterwards.
* **Casting:** `str()`, `int()`, `float()`, `bool()`; inspect with `type()` / `isinstance()`.
* **Constants:** `typing.Final` – enforced by type checkers only, not at runtime.

In [ ]:
from typing import Final

age: int = 10
greeting: str = "Hello"
is_active: bool = True
names: list[str] = ["Bob", "Anna", "Luigi"]
point: tuple[float, float] = (1.5, 2.5)
unique_ids: set[int] = {1, 4, 2, 9}
user: dict[str, object] = {"name": "Bob", "age": 20}
VERSION: Final[str] = "1.0.1"

x, y, z = "Orange", "Banana", "Cherry"   # multiple assignment
a = b = c = 0                            # same value
first, *middle, last = [1, 2, 3, 4, 5]   # extended unpacking

print(str(3), int("3"), float(3), type(greeting))
print(x, y, z, first, middle, last)
print(isinstance(3.14, int | float))     # union in isinstance – Python 3.10+

try:
    print(age + greeting)
except TypeError as e:
    print("TypeError:", e)               # unsupported operand type(s) for +: 'int' and 'str'

### ⚡ PySpark – Python types ↔ Spark types
| Python | Spark SQL type |
|---|---|
| `int` | `LongType` (`bigint`) |
| `float` | `DoubleType` |
| `str` | `StringType` |
| `bool` | `BooleanType` |
| `list` | `ArrayType` |
| `dict` | `MapType` / `StructType` |
| `datetime.date` / `datetime.datetime` | `DateType` / `TimestampType` |

Casting in Spark is done with `col.cast("type")`. 🆕 In **Spark 4.0 ANSI mode is on by default** – an invalid cast raises an error instead of silently returning `NULL`; use `try_cast` when bad input is expected.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("age", LongType(), nullable=True),
    StructField("score", StringType(), nullable=True),   # deliberately a string
])

people_df = spark.createDataFrame(
    [("Bob", 20, "95.5"), ("Anna", 31, "88"), ("Luigi", None, "n/a")],
    schema=schema,
)

typed_df = people_df.withColumn("score_double", F.expr("try_cast(score AS DOUBLE)"))
typed_df.printSchema()
display(typed_df)

## 2. Strings

* Quotes: `'...'`, `"..."`, multi-line `"""..."""` (keeps line breaks). Escapes: `\'`, `\"`, `\n`, `\t`. Raw strings: `r"C:\Users\..."`.
* **f-strings:** `{x:,}` thousands separator, `{x:.2f}` 2 decimals, `{x:.2%}` percent, `{x=}` debug form.
* **Corrections to the notes:**
  * `strip()/lstrip()/rstrip()` remove a **set of characters**, not a substring → use `removeprefix()/removesuffix()` for substrings.
  * `rfind()` returns the **last** (highest) index; `find()` → `-1`, `index()` → `ValueError` when missing.
  * `expandtabs()` replaces `\t` with **spaces** up to the next tab stop.
* Build long strings with `"".join(parts)` – not `+=` in a loop.

🆕 **Python 3.14 – template strings (PEP 750):** `t"Hello {name}"` returns a `string.templatelib.Template` object (not a `str`), so libraries can safely escape the interpolated values (SQL, HTML, shell).

In [ ]:
text: str = "  Hello, World!  "
print(repr(text.strip()))                          # 'Hello, World!'
print("abcab".strip("ab"))                         # 'c'  -> character set!
print("prefix_data".removeprefix("prefix_"))       # 'data' -> substring
print("banana".find("a"), "banana".rfind("a"))     # 1 5
print("Hello"[::-1])                               # 'olleH'
print("A\tB".expandtabs(4) + "|")                  # 'A   B|'
print(f"{123456:,} | {1234.567:.2f} | {0.5:.2%}")  # 123,456 | 1234.57 | 50.00%
value = 42
print(f"{value=}")                                 # value=42

parts: list[str] = []
for i in range(3):
    parts.append(f"line {i}")
print("\n".join(parts))

# list of public str methods (from the notes, page 4)
public_methods = [m for m in dir(str) if not m.startswith("_")]
print(len(public_methods), public_methods[:8], "...")

In [ ]:
if PY_314:
    # compiled only on Python 3.14+, so older runtimes do not fail
    exec('name = "Bob"; tpl = t"Hello {name}!"; print(type(tpl).__name__, tpl.strings, tpl.values)')
else:
    print("t-strings need Python 3.14+ – current:", sys.version.split()[0])

### ⚡ PySpark – string functions
Spark has built-in, JVM-optimised equivalents of Python string methods – prefer them over Python UDFs.

| Python | PySpark (`pyspark.sql.functions`) |
|---|---|
| `s.upper()` / `s.lower()` / `s.title()` | `F.upper` / `F.lower` / `F.initcap` |
| `s.strip()` | `F.trim` / `F.ltrim` / `F.rtrim` |
| `s.replace(a, b)` | `F.replace` / `F.regexp_replace` |
| `s.split(",")` | `F.split` |
| `s.startswith(x)` | `F.col(...).startswith(x)` / `F.startswith` |
| `s[::-1]` | `F.reverse` |
| `f"{x:,.2f}"` | `F.format_number(col, 2)` |
| `",".join(lst)` | `F.concat_ws(",", ...)` / `F.array_join` |

In [ ]:
str_df = spark.createDataFrame([("  mario rossi ", 1234.567), ("james BOND", 0.5)], "raw string, amount double")

display(
    str_df.select(
        F.initcap(F.trim("raw")).alias("title_case"),
        F.upper("raw").alias("upper"),
        F.reverse(F.trim("raw")).alias("reversed"),
        F.split(F.trim("raw"), " ").alias("tokens"),
        F.regexp_replace("raw", r"\s+", "_").alias("snake"),
        F.format_number("amount", 2).alias("amount_fmt"),
        F.col("raw").contains("BOND").alias("is_bond"),
    )
)

## 3. Operators & control flow

* Arithmetic: `//` **floor** division (rounds toward −∞: `-7 // 2 == -4`), `%` remainder, `**` power, `x += 2`.
* `is` = identity (same object), `==` = equality. Compare floats with `math.isclose`.
* **Truthiness** – falsy: `None`, `False`, `0`, `0.0`, `""`, `[]`, `()`, `{}`, `set()`; everything else is truthy. Write `if condition:`, not `if condition == True:`.
* `if / elif / else`, **ternary** `a if cond else b`, `for` + `range(start, stop, step)` (stop exclusive), `while`, `break`, `continue`.
* **`match` / `case`** (3.10+) – structural pattern matching; `_` is the wildcard. Its benefit is readability and destructuring, not speed.
* **Guard clause** – return early on invalid input instead of nesting `if/else`.

In [ ]:
import math

print(7 // 2, -7 // 2, 10 % 3, 2 ** 4)               # 3 -4 1 16
print(math.isclose(0.999, 1.000, abs_tol=0.002))     # True
print(0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3)) # False True

volume: float = 0.3
loudness: str = "LOUD" if volume > 0.6 else "SILENT"
print(loudness)

def http_status(status: int) -> str:
    match status:
        case 200:
            return "Connected"
        case 403 | 401:
            return "Forbidden"
        case 404:
            return "Not found"
        case _:
            return "Unknown error"

def parse_command(cmd: str) -> str:
    match cmd.split():
        case ["rename", *files] if files:        # literal + star pattern + guard
            return f"rename {len(files)} file(s): {files}"
        case ["quit" | "exit"]:
            return "bye"
        case _:
            return "unknown command"

print([http_status(s) for s in (200, 401, 404, 500)])
print(parse_command("rename a.txt b.txt"), "|", parse_command("exit"))

def start_engine(has_fuel: bool, is_on: bool, battery_ok: bool) -> str:
    # guard clauses
    if not has_fuel:
        return "No fuel"
    if is_on:
        return "Already running"
    if not battery_ok:
        return "Battery problem"
    return "Engine started"

print(start_engine(True, False, True))

i = 5
while i > 0:
    i -= 1
    if i == 3:
        continue
    if i == 1:
        break
    print(f"{i} - hello")

### ⚡ PySpark – conditional logic
* `if/elif/else` → `F.when(...).when(...).otherwise(...)`
* `match/case` on values → chained `F.when` or SQL `CASE WHEN`
* 🆕 Spark 4 ANSI mode: division by zero **raises** – use `F.try_divide` (Spark 3.5+) when zeros are expected.

In [ ]:
status_df = spark.createDataFrame([(200, 10, 2), (404, 5, 0), (500, 7, 7)], "status int, num int, den int")

display(
    status_df.select(
        "status",
        F.when(F.col("status") == 200, "Connected")
         .when(F.col("status").isin(401, 403), "Forbidden")
         .when(F.col("status") == 404, "Not found")
         .otherwise("Unknown error").alias("message"),
        F.try_divide("num", "den").alias("safe_ratio"),          # NULL instead of an error
        F.expr("num div 2").alias("floor_div"),                   # integer division in SQL
        (F.col("num") % 3).alias("remainder"),
    )
)

## 4. Functions, scope, lambdas & decorators

* `def name(param: type = default) -> return_type:` – without `return` a function returns `None`.
* **Parameter** = name in the definition; **argument** = value passed in the call. Positional arguments go before keyword arguments.
* `/` → everything before is **positional-only**; `*` → everything after is **keyword-only**.
* `*args` collects extra positional arguments into a **tuple**; `**kwargs` collects extra keyword arguments into a **dict**. At call-site `f(*lst, **dct)` unpacks.
* **Scope (LEGB):** Local → Enclosing → Global → Built-in. `global` rebinds a module-level name, `nonlocal` rebinds a name in the enclosing function. Avoid shadowing.
* **Placeholders:** `...`, `pass`, best: `raise NotImplementedError("...")`.
* **Lambda** – anonymous one-expression function, for inline use (PEP 8: use `def` when you need a name).
* **Decorator** – a function that takes a function and returns a (wrapped) function; use `functools.wraps` and **return the wrapped result**.

In [ ]:
from collections.abc import Callable
from functools import wraps
from time import perf_counter, sleep

def greet(name: str, /, lang: str = "en", *, default: str = "Hello") -> str:
    return f"Ciao, {name}!" if lang == "it" else f"{default}, {name}!"

def add(*args: int) -> int:
    return sum(args)

def pin_position(**kwargs: int) -> dict[str, int]:
    return kwargs

print(greet("Mario", "it"), greet("Bob", default="Hi"))
print(add(1, 2, 3), pin_position(x=10, y=20))
print(add(*[4, 5]), greet(*["Luigi"], **{"lang": "it"}))

counter: int = 0
def increment() -> None:
    global counter
    counter += 1

def outer() -> str:
    name = ""
    def inner() -> None:
        nonlocal name
        name = "Tom"
    inner()
    return name

increment()
print(counter, outer())

def placeholder() -> None:
    raise NotImplementedError("placeholder() is not ready yet")

def timed(func: Callable) -> Callable:
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)          # keep the return value!
        print(f"{func.__name__}() took {perf_counter() - start:.3f}s")
        return result
    return wrapper

@timed
def calculate(a: int, b: int) -> int:
    sleep(0.2)
    return a + b

print(calculate(1, 2), calculate.__name__)

def use_all(f: Callable[[int], None], values: list[int]) -> None:
    for v in values:
        f(v)

use_all(lambda v: print("X" * v), [2, 4])

### ⚡ PySpark – functions on columns: built-in → pandas UDF → Python UDF
1. **Built-in functions** (`F.*`) – fastest, run in the JVM, optimised by Catalyst. Always try these first.
2. **pandas UDF** (`@F.pandas_udf`) – vectorised, data moves in Arrow batches.
3. **Python UDF** (`@F.udf`) – row-by-row; 🆕 `useArrow=True` (Spark 3.5+) makes it Arrow-optimised.

A UDF is literally a **decorator** applied to a Python function – the decorator concept from the notes in action.

In [ ]:
import pandas as pd

@F.pandas_udf("double")
def to_celsius(f: pd.Series) -> pd.Series:
    return (f - 32) * 5.0 / 9.0

@F.udf(returnType="string", useArrow=True)
def describe(temp_c: float) -> str:
    return "hot" if temp_c is not None and temp_c > 25 else "mild"

temps_df = spark.createDataFrame([("Rome", 86.0), ("Oslo", 50.0)], "city string, temp_f double")

result_df = (
    temps_df
    .withColumn("temp_c_builtin", F.round((F.col("temp_f") - 32) * 5 / 9, 1))  # 1) built-in
    .withColumn("temp_c_pandas", to_celsius("temp_f"))                         # 2) pandas UDF
    .withColumn("label", describe("temp_c_pandas"))                            # 3) Python UDF
)
display(result_df)

## 5. Collections & built-in functions

| Type | Mutable | Ordered | Duplicates | Literal |
|---|---|---|---|---|
| `list` | ✅ | ✅ | ✅ | `[1, 2]` |
| `tuple` | ❌ | ✅ | ✅ | `1, 2` or `(1, 2)` |
| `set` | ✅ | ❌ (not subscriptable) | ❌ | `{1, 2}`, empty: `set()` |
| `frozenset` | ❌ | ❌ | ❌ | `frozenset({1, 2})` |
| `dict` | ✅ | ✅ insertion order (3.7+) | keys unique | `{"k": 1}`, empty: `{}` |

* **Slicing** `seq[start:stop:step]` – index from 0, `stop` exclusive, `[::-1]` reverses.
* `set.pop()` removes an **arbitrary** element. Never modify a list while iterating over it.
* Built-ins: `enumerate(start=)`, `zip(strict=)` 🆕 3.10, `range` (lazy – wrap in `list()` to print), `slice`, `sorted(key=)`, `max/min(key=)`, `all`, `any`, `round` (returns float when `ndigits` is given), `callable`, `isinstance`, `globals`, `locals`.
* ⚠️ `eval()` / `exec()` run arbitrary code – never use them on user input.

In [ ]:
people: list[str] = ["Bob", "James", "Tom"]
people.append("Jeremy"); people.insert(1, "Timothy"); people.remove("Bob"); people[0] = "Charlotte"
print(people)                                            # ['Charlotte', 'James', 'Tom', 'Jeremy']
removed = people.pop()                                   # removes the last item
print(removed, people)

numbers: list[int] = [1, 2, 3, 4, 5, 6]
print(numbers[0:3], numbers[3:], numbers[0:4:2], numbers[4:0:-2], numbers[::2], numbers[::-1])

elements: set = {99, True, "Bob"}
elements.add("James"); elements.discard("Bob")          # discard: no error if missing
print(elements, frozenset({99, "Bob"}))

weather: dict = {"time": "12:00", "weather": {"morning": "rain", "evening": "more rain"}}
print(weather["weather"]["morning"], weather.get("missing", "default"))

users: dict[int, str] = {1: "Luigi", 2: "Bob"}
users[3] = "Mario"; users[1] = "James"; users.pop(2)
print(users, users | {4: "Peach"})                       # dict merge operator (3.9+)

print(list(enumerate(["A", "B", "C"], start=1)))
print(list(zip(["a", "b", "c"], [1, 2])))               # truncates to shortest
try:
    list(zip(["a", "b", "c"], [1, 2], strict=True))
except ValueError as e:
    print("strict zip:", e)

print(list(range(1, 6)), list(range(-5, 0)), list(range(-1, -6, -1)))
print(round(265.56, -1), round(2.5), round(3.5))         # 270.0 2 4  (banker's rounding)
print("Hello"[slice(0, 3)], "Hello"[slice(None, None, -1)])

scores = [{"name": "Bob", "score": 20}, {"name": "Sandra", "score": 95}]
print(max(scores, key=lambda s: s["score"]))
print(all([True, 1, "x"]), any([0, "", None, 5]))
print(callable(range), callable("string"))

class Animal: ...
class Cat(Animal): ...
print(isinstance(Cat(), Animal), isinstance(Animal(), Cat))

### ⚡ PySpark – complex types
Python `list` → `ArrayType`, `dict` → `MapType`, nested objects → `StructType`.
Useful functions: `F.explode` (one row per element – like a `for` loop), `F.collect_list` / `F.collect_set` (inverse – `set` semantics removes duplicates), `F.size`, `F.array_contains`, `F.element_at`, `F.map_keys`, `F.posexplode` (like `enumerate`), `F.arrays_zip` (like `zip`).

In [ ]:
nested_df = spark.createDataFrame(
    [("Mario", ["pizza", "pasta"], {"level": 3}), ("Luigi", ["pasta"], {"level": 2})],
    "name string, foods array<string>, stats map<string,int>",
)

display(nested_df.select("name", F.size("foods").alias("n_foods"), F.col("stats")["level"].alias("level")))
display(nested_df.select("name", F.posexplode("foods").alias("pos", "food")))       # enumerate()
display(nested_df.select(F.explode("foods").alias("food")).agg(F.collect_set("food").alias("unique_foods")))

## 6. Comprehensions & functional tools

**Structure:** `[return_value  for item in iterable  if condition]`
* list `[...]`, set `{...}`, dict `{k: v ...}`, generator `(...)`.
* `map(func, iterable)` and `filter(func, iterable)` return **lazy iterators** (memory-efficient); a list comprehension is evaluated immediately.
* `sorted(iterable, key=...)` is stable – equal keys keep their original order.

In [ ]:
names: list[str] = ["Mario", "James", "Luigi", "John", "Charlotte"]

j_names = [n for n in names if n.startswith("J")]
upper_names = [n.upper() for n in names]                 # the fixed version of upper_everything()
name_len = {n: len(n) for n in names}
initials = {n[0] for n in names}
doubled = list(map(lambda n: n * 2, [1, 2, 3]))
evens = list(filter(lambda n: n % 2 == 0, range(1, 21)))
by_length = sorted(["Mario", "James", "Anna", "Tom"], key=len)

print(j_names, upper_names, name_len, initials, doubled, evens, by_length, sep="\n")

### ⚡ PySpark – higher-order functions (comprehensions inside a column)
`F.transform` ≈ list comprehension / `map`, `F.filter` ≈ `filter`, `F.aggregate` ≈ `functools.reduce`, `F.exists` / `F.forall` ≈ `any` / `all`.
The Python lambda is translated to a Spark SQL lambda – it runs in the JVM, no UDF needed.

In [ ]:
arr_df = spark.createDataFrame([([1, 2, 3, 4, 5],), ([10, 15],)], "nums array<int>")

display(
    arr_df.select(
        "nums",
        F.transform("nums", lambda x: x * 2).alias("doubled"),
        F.filter("nums", lambda x: x % 2 == 0).alias("evens"),
        F.aggregate("nums", F.lit(0), lambda acc, x: acc + x).alias("total"),
        F.exists("nums", lambda x: x > 10).alias("any_gt_10"),
        F.forall("nums", lambda x: x > 0).alias("all_positive"),
        F.array_sort("nums", lambda l, r: r - l).alias("desc"),
    )
)

## 7. Generators & iterators

* A **generator** function uses `yield`; values are produced **on demand** and **consumed once**.
* More memory-efficient than lists (not necessarily faster).
* 🆕 Type hints: import from `collections.abc` (the `typing.Generator` alias is deprecated since 3.9). Since Python 3.13 `Generator[int]` = `Generator[int, None, None]`; `Iterator[int]` is the simplest annotation.

In [ ]:
from collections.abc import Iterator
from itertools import islice

def fibonacci() -> Iterator[int]:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print(list(islice(fib, 10)))          # first 10
print(next(fib), next(fib))           # continues where it stopped: 55 89

squares = (n * n for n in range(5))   # generator expression
print(sum(squares), sum(squares))     # 30 0  -> consumed once!

### ⚡ PySpark – generators in `mapInPandas`
`DataFrame.mapInPandas(func, schema)` passes an **iterator of pandas DataFrames** (one per Arrow batch) to a generator function – the generator pattern from the notes is how you process a big Spark table batch by batch in Python.

In [ ]:
def add_fibonacci_flag(batches: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    fib_set = set(islice(fibonacci(), 40))
    for pdf in batches:
        pdf["is_fib"] = pdf["id"].isin(fib_set)
        yield pdf

display(spark.range(0, 25).mapInPandas(add_fibonacci_flag, schema="id long, is_fib boolean"))

## 8. Object-oriented programming & dataclasses

* **Class** = blueprint, **object/instance** = thing built from it, **method** = function defined in a class, `self` = current instance, `__init__` = initialiser.
* **Class attribute** – shared by all instances (don't use mutable class attributes such as lists). **Instance attribute** – per object.
* **Inheritance** + `super().__init__(...)`.
* `@staticmethod` (no `self`/`cls`), `@classmethod` (receives `cls`), `@abstractmethod` (needs `abc.ABC`; subclasses must implement it before they can be instantiated), `@property` (computed attribute), `functools.cached_property` (computed once).
* **Dunder methods:** `__str__` (for users), `__repr__` (for developers), `__eq__` (return `NotImplemented` for other types).
* **Name mangling:** `__attr` → `_ClassName__attr` – avoids clashes in subclasses; not real privacy.
* `@dataclass` generates `__init__`, `__repr__`, `__eq__`; `__post_init__` runs after init; `InitVar` = init-only value passed to `__post_init__`, **not stored**. 🆕 `slots=True`, `kw_only=True` (3.10+).
* `Enum` – fixed set of named constants (`StrEnum` 3.11+).

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, InitVar, asdict
from enum import StrEnum
from functools import cached_property
from typing import Self

class Shape(ABC):
    sides_label: str = "sides"                 # class attribute

    def __init__(self, name: str, sides: int) -> None:
        self.name = name
        self.sides = sides
        self.__secret = "mangled"              # -> self._Shape__secret

    @abstractmethod
    def area(self) -> float: ...

    @staticmethod
    def describe_sides(n: int) -> str:
        return f"{n} sides"

    @classmethod
    def unit(cls) -> Self:
        return cls("unit", 4)                  # type: ignore[call-arg]

    def __str__(self) -> str:
        return f"{self.name} ({self.sides} {self.sides_label})"

    def __repr__(self) -> str:
        return f"{type(self).__name__}(name={self.name!r}, sides={self.sides})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Shape):
            return NotImplemented
        return (self.name, self.sides) == (other.name, other.sides)

class Square(Shape):
    def __init__(self, name: str, sides: int = 4, length: float = 1.0) -> None:
        super().__init__(name, sides)
        self.length = length

    def area(self) -> float:
        return self.length ** 2

    @cached_property
    def diagonal(self) -> float:
        return self.length * 2 ** 0.5

sq = Square("sq", length=3)
print(sq, repr(sq), sq.area(), round(sq.diagonal, 3), Shape.describe_sides(4))
print(sq == Square("sq"), Square.unit(), sq._Shape__secret)
try:
    Shape("abstract", 0)
except TypeError as e:
    print("TypeError:", e)

class Unit(StrEnum):
    GRAM = "g"
    KILOGRAM = "kg"

@dataclass(slots=True)
class Fruit:
    name: str
    grams: float
    price_per_kg: float
    unit: Unit = Unit.GRAM
    tags: list[str] = field(default_factory=list)   # safe mutable default
    is_rare: InitVar[bool | None] = None

    def __post_init__(self, is_rare: bool | None) -> None:
        if is_rare:
            self.tags.append("rare")

    @property
    def total_price(self) -> float:                  # recomputed on every access
        return round(self.grams / 1000 * self.price_per_kg, 2)

apple = Fruit("apple", 250, 4.0, is_rare=True)
print(apple, apple.total_price, apple == Fruit("apple", 250, 4.0, tags=["rare"]))

### ⚡ PySpark – dataclasses → DataFrame (and back)
`dataclasses.asdict()` turns objects into dicts that `spark.createDataFrame` understands; `Row.asDict()` goes the other way.

In [ ]:
fruits = [Fruit("apple", 250, 4.0), Fruit("mango", 400, 12.5, is_rare=True), Fruit("kiwi", 90, 9.0)]

fruit_rows = [asdict(f) | {"total_price": f.total_price, "unit": str(f.unit)} for f in fruits]
fruit_df = spark.createDataFrame(fruit_rows)
display(fruit_df)

back = [Fruit(r["name"], r["grams"], r["price_per_kg"], tags=list(r["tags"])) for r in fruit_df.limit(2).collect()]
print(back)

## 9. Modules, packages & imports

* **Module** – a single `.py` file. **Package** – a directory of modules. **Library** – a collection of packages/modules.
* **CONTENT GOT OUTDATED!** – the note "a package *must* contain `__init__.py`" is outdated since **Python 3.3 (PEP 420, namespace packages)**. `__init__.py` is still recommended for regular packages.
* Prefer explicit imports; with `from x import *` the **last import wins** on name clashes.
* `if __name__ == "__main__": main()` – code runs only when the file is executed, not when it is imported.
* Spark convention: `from pyspark.sql import functions as F`, `from pyspark.sql import types as T` (avoids clashes with Python's `sum`, `max`, `filter`, …).
* 🆕 **Python 3.15 – explicit lazy imports (PEP 810):** `lazy import json` defers loading the module until first use (faster start-up of CLIs/notebooks).
* 🆕 **Databricks:** `.py` files in Git folders / workspace files can be imported directly from notebooks (`from utils.helpers import clean`); `%pip install` for notebook-scoped libraries; `uv`-based dependency management is common outside Databricks.

In [ ]:
import builtins
from pyspark.sql import functions as F, types as T

print(builtins.max is max, F.max is max)   # True False -> why "import *" from pyspark is dangerous
print(__name__)                            # "__main__" in a notebook

def main() -> None:
    print("Execution...")

if __name__ == "__main__":
    main()

## 10. Type hints & tooling

* Annotations are **hints**, not runtime checks (`x: int = 10.0` runs fine – only the IDE / checker complains).
* Modern syntax: `list[str]`, `dict[str, int]`, `int | None` (3.10), `Self` (3.11), `type Alias = ...` and generics `def first[T](xs: list[T]) -> T` (3.12, PEP 695).
* **CONTENT GOT OUTDATED!** – `from typing import Callable, Generator, List` are deprecated aliases → use `collections.abc` / built-ins.
* 🆕 **Python 3.14 (PEP 649/749):** annotations are evaluated **lazily**; forward references work without quotes; inspect with `annotationlib.get_annotations()`.
* **Checkers:** `mypy file.py` (from the notes) – still valid; 🆕 faster alternatives: `pyright`, Astral `ty`, Meta `pyrefly`; `ruff` for lint + format.

In [ ]:
def first[T](items: list[T]) -> T:          # PEP 695 generics (Python 3.12+)
    return items[0]

type Matrix = list[list[float]]              # PEP 695 type alias

def upper_everything(elements: list[str]) -> list[str]:
    return [e.upper() for e in elements]

number: int = 10.0                            # type: ignore[assignment]  <- runs fine, checker would flag it
print(first(["a", "b"]), upper_everything(["x", "y"]), number, upper_everything.__annotations__)

if PY_314:
    import annotationlib
    print(annotationlib.get_annotations(upper_everything, format=annotationlib.Format.STRING))

## 11. Files & JSON

* Always use a **context manager**: `with open(path, mode, encoding="utf-8") as f:` – the file is closed even on errors (the notes' "memory leakage" is really a *file-handle* leak).
* Modes: `r` read, `w` write (**truncates**), `a` append (creates if missing), `x` create-only, `b` binary. Missing file → `FileNotFoundError`.
* `read()`, `readline()` (returns `''` at EOF), `readlines()`, `write()`, `writelines()`.
* **CONTENT GOT OUTDATED!** – relying on the default encoding: **Python 3.15 (PEP 686) enables UTF-8 mode by default**; older versions use the OS locale (cp1250 on Polish Windows). Pass `encoding="utf-8"` explicitly.
* `json.load(file)` / `json.loads(str)` → Python objects; `json.dump(obj, file)` / `json.dumps(obj)` → JSON.
* 🆕 **Databricks:** store files in **Unity Catalog volumes** (`/Volumes/<catalog>/<schema>/<volume>/...`); the driver's `/tmp` is local and ephemeral.

In [ ]:
import json
import tempfile
from pathlib import Path

work_dir = Path(tempfile.mkdtemp())
txt_path = work_dir / "info.txt"

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("I'm some text!\n")
with open(txt_path, "a", encoding="utf-8") as f:
    f.writelines(["Eggs\n", "Ham\n", "Zażółć gęślą jaźń\n"])

with open(txt_path, encoding="utf-8") as f:
    print(f.readline().rstrip())
    print(f.readlines())

json_path = work_dir / "new_file.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({"name": "Bob", "age": 45, "langs": ["pl", "en"]}, f, ensure_ascii=False)

with open(json_path, encoding="utf-8") as f:
    print(json.load(f))
print(json.loads('{"name": "Mario", "age": 33}'))

try:
    open(work_dir / "missing.txt", encoding="utf-8")
except FileNotFoundError as e:
    print("FileNotFoundError:", e.strerror)

### ⚡ PySpark – JSON
* Files: `spark.read.json("/Volumes/...")` / `df.write.json(...)` (use a UC volume path on your workspace).
* JSON stored as **strings** in a column: `F.from_json(col, schema)`, `F.get_json_object`, `F.to_json`.
* 🆕 **Spark 4.0 / DBR 15.3+: `VARIANT` type** – `F.parse_json` stores semi-structured data without a fixed schema; query with `F.variant_get` or the `:` path syntax in SQL.

In [ ]:
raw_json_df = spark.createDataFrame(
    [('{"name": "Mario", "age": 33, "skills": ["jump"]}',), ('{"name": "Bob", "age": 45}',)],
    "payload string",
)

parsed_df = raw_json_df.select(
    F.from_json("payload", "name string, age int, skills array<string>").alias("j")
).select("j.*")
display(parsed_df)

if hasattr(F, "parse_json"):
    variant_df = raw_json_df.select(F.parse_json("payload").alias("v"))
    display(variant_df.select(
        F.variant_get("v", "$.name", "string").alias("name"),
        F.variant_get("v", "$.age", "int").alias("age"),
    ))
else:
    print("VARIANT requires Spark 4.0+ / DBR 15.3+")

## 12. Errors & exceptions

* `try` → `except SomeError as e` → `else` (no exception) → `finally` (always runs).
* `raise ValueError("message")` to signal problems; custom exceptions subclass `Exception`.
* `assert cond, "msg"` – for invariants/tests only; asserts are **removed** with `python -O`.
* In notebooks avoid `sys.exit()` – use `dbutils.notebook.exit("value")` in Databricks.
* 🆕 3.11: `ExceptionGroup` + `except*`, `e.add_note()`. 🆕 3.14 (PEP 758): `except ValueError, TypeError:` without parentheses; (PEP 765) `return`/`break`/`continue` that exit a `finally` block emit `SyntaxWarning`.
* PySpark errors live in `pyspark.errors` (`AnalysisException`, `PySparkValueError`, …) and carry an **error class** / SQLSTATE.

In [ ]:
class ValidationError(Exception):
    """Raised when input data is invalid."""

def parse_number(user_input: str) -> float:
    try:
        value = float(user_input)
    except ValueError as e:
        e.add_note(f"input was {user_input!r}")
        raise ValidationError("not a number") from e
    else:
        return value
    finally:
        print(f"parse_number({user_input!r}) finished")

for candidate in ("3.14", "abc"):
    try:
        print(parse_number(candidate))
    except ValidationError as e:
        print(type(e).__name__, "-", e, "| cause:", repr(e.__cause__), e.__cause__.__notes__)

try:
    raise ExceptionGroup("batch failed", [ValueError("bad row 1"), KeyError("missing col")])
except* ValueError as eg:
    print("value errors:", [str(x) for x in eg.exceptions])
except* KeyError as eg:
    print("key errors:", [str(x) for x in eg.exceptions])

In [ ]:
from pyspark.errors import AnalysisException

try:
    spark.range(3).select("does_not_exist").collect()
except AnalysisException as e:
    # Spark 4.0 renamed "error class" to "condition"; getErrorClass() is kept as a deprecated alias
    condition = e.getCondition() if hasattr(e, "getCondition") else e.getErrorClass()
    print("Condition  :", condition)
    print("SQLSTATE   :", e.getSqlState())

## 13. Async IO

* `async def` defines a **coroutine**; `await` suspends it until the awaited operation finishes – good for I/O-bound work (HTTP calls, DB queries), not for CPU-bound work.
* **Task** – a scheduled coroutine (`asyncio.create_task`) that can be awaited, cancelled or time-limited (`asyncio.timeout`, 3.11+).
* **Gather** *(left blank in the notes)* – `await asyncio.gather(*coros)` runs awaitables **concurrently** and returns results **in input order**; `return_exceptions=True` collects errors instead of failing fast.
* 🆕 **`asyncio.TaskGroup`** (3.11+) – structured concurrency: if one task fails the others are cancelled; preferred over `gather` in new code.
* Script entry point: `asyncio.run(main())`. **In notebooks (Jupyter/Databricks) an event loop is already running** → `asyncio.run()` raises `RuntimeError`; use top-level `await main()` or run the loop in a worker thread (shown below, works on every runtime).

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def fetch(source: str, delay: float) -> str:
    await asyncio.sleep(delay)          # stands for an I/O call
    return f"{source} done in {delay}s"

async def main() -> dict[str, list[str]]:
    gathered = await asyncio.gather(fetch("api", 0.3), fetch("db", 0.1), fetch("s3", 0.2))
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(fetch(f"job-{i}", 0.1 * i)) for i in range(1, 4)]
    return {"gather": gathered, "taskgroup": [t.result() for t in tasks]}

start = perf_counter()
with ThreadPoolExecutor(max_workers=1) as pool:
    results = pool.submit(asyncio.run, main()).result()
print(results, f"\ntotal: {perf_counter() - start:.2f}s (concurrent, not 1.2s sequential)")

## 14. Pandas API on Spark

`pyspark.pandas` gives a **pandas-like API executed by Spark** (distributed, no driver-memory limit).

| Conversion | Code |
|---|---|
| Spark DF → pandas-on-Spark | `sdf.pandas_api()` or `ps.DataFrame(sdf)` |
| pandas-on-Spark → Spark DF | `psdf.to_spark()` |
| Spark DF → local pandas (small data only!) | `sdf.toPandas()` |

**CONTENT GOT OUTDATED!** – the notes use `spark_df.to_pandas_on_spark()`. It was **deprecated in Spark 3.3 and removed in Spark 4.0** (Databricks Runtime 17+). Use `spark_df.pandas_api()`.

In [ ]:
try:
    import pyspark.pandas as ps

    sdf = spark.createDataFrame([("Bob", 20, 1.5), ("Anna", 31, 2.5), ("Luigi", 27, 3.0)], "name string, age int, score double")
    psdf = sdf.pandas_api()                     # modern replacement of to_pandas_on_spark()
    psdf["score_x2"] = psdf["score"] * 2
    print(psdf[psdf["age"] > 21].sort_values("age"))
    print(psdf.describe())

    display(psdf.to_spark())                    # back to a Spark DataFrame
except Exception as e:                          # pandas API on Spark may be limited on some compute types
    print("Pandas API on Spark not available here:", type(e).__name__, e)

## 15. What's new in Spark 4.x & Databricks (not in the original notes)

| Feature | Since | Why it matters |
|---|---|---|
| **ANSI SQL mode on by default** | Spark 4.0 | Overflow, invalid casts, division by zero raise errors → use `try_*` functions |
| **`VARIANT` type** (`parse_json`) | Spark 4.0 | Fast semi-structured data without schema |
| **SQL pipe syntax `\|>`** | Spark 4.0 | Write SQL top-to-bottom like DataFrame chains |
| **Python Data Source API** | Spark 4.0 | Custom readers/writers in pure Python (`pyspark.sql.datasource`) |
| **Python UDTFs** (`@udtf`) | Spark 3.5 | Table-valued functions written in Python (generator `yield` rows!) |
| **Native plotting `df.plot`** | Spark 4.0 | Plotly-based charts directly from DataFrames |
| **Spark Connect** | Spark 3.4+, default on serverless | Thin client; no `sparkContext`/RDD |
| **Spark Declarative Pipelines / Lakeflow Declarative Pipelines** (formerly *Delta Live Tables*) | Databricks 2025, Spark 4.1 | Declarative Medallion pipelines |
| **Unity Catalog volumes** | Databricks | Governed file storage (`/Volumes/...`) |
| **Python 3.13–3.15** | 2024–2026 | Free-threaded build officially supported (3.14, PEP 779), experimental JIT (3.13), t-strings (3.14), lazy imports & UTF-8 default (3.15) |

In [ ]:
try:
    display(spark.sql("""
        FROM range(10)
        |> WHERE id % 2 = 0
        |> EXTEND id * id AS squared
        |> SELECT id, squared
    """))
except Exception as e:
    print("SQL pipe syntax requires Spark 4.0+ / DBR 16.2+:", type(e).__name__)

In [ ]:
from pyspark.sql.functions import udtf

@udtf(returnType="word string, position int")
class SplitWords:
    def eval(self, text: str):               # a generator – yields rows
        for i, word in enumerate(text.split(), start=1):
            yield word, i

display(SplitWords(F.lit("Python notes to Databricks")))

## 16. Capstone – Medallion pipeline in PySpark

The same architecture used to produce this notebook (bronze → silver → gold → platinum), expressed with Spark DataFrames:
* **Bronze** – raw ingested records, as-is (here: topics per handwritten page, with duplicates and typos).
* **Silver** – cleaned, typed, standardised and **deduplicated** (`dropDuplicates`).
* **Gold** – business-level aggregates ready for consumption.

In production each layer is a **Delta table** in Unity Catalog (`df.write.saveAsTable("catalog.schema.table")`) or a Lakeflow Declarative Pipeline; here temporary views keep the demo self-contained.

In [ ]:
bronze_df = spark.createDataFrame(
    [
        ("python-03.pdf", 3, "List comprehensions "), ("python-10.pdf", 10, "list Comprehensions"),
        ("python-03.pdf", 3, "mypy"), ("python-16.pdf", 16, "MyPy"),
        ("python-03.pdf", 3, "Generators"), ("python-17.pdf", 17, "generators"),
        ("python-09.pdf", 9, "Packages"), ("python-18.pdf", 18, "Pandas API on Spark"),
        ("python-15.pdf", 15, "zip"), ("python-17.pdf", 17, "Match-case"),
    ],
    "source_file string, page int, topic string",
).withColumn("_ingested_at", F.current_timestamp())
bronze_df.createOrReplaceTempView("bronze_notes")

outdated_topics = ["packages", "pandas api on spark", "zip"]

silver_df = (
    spark.table("bronze_notes")
    .withColumn("topic", F.lower(F.trim("topic")))
    .withColumn("topic", F.regexp_replace("topic", r"^mypy$", "mypy (static typing)"))
    .groupBy("topic")
    .agg(F.sort_array(F.collect_set("page")).alias("pages"), F.min("page").alias("first_page"))
    .withColumn("is_outdated", F.col("topic").isin(outdated_topics))
)
silver_df.createOrReplaceTempView("silver_notes")

gold_df = spark.sql("""
    SELECT
        count(*)                                         AS unique_topics,
        sum(size(pages))                                 AS raw_mentions,
        sum(CASE WHEN size(pages) > 1 THEN 1 ELSE 0 END) AS duplicated_topics,
        sum(CASE WHEN is_outdated THEN 1 ELSE 0 END)     AS outdated_topics
    FROM silver_notes
""")

display(silver_df.orderBy("first_page"))
display(gold_df)

---
### Sources of this notebook
* `bronze_01.md` … `bronze_19.md` – OCR of the handwritten pages
* `silver.md` – aggregated OCR
* `gold.md` – discipline analysis, correctness review, deduplication, "CONTENT GOT OUTDATED!" findings
* Enrichment: Python 3.10–3.15 release notes and PEPs (585, 604, 618, 634, 649, 654, 686, 695, 696, 750, 758, 765, 779, 810), Apache Spark 3.5 / 4.x release notes, Databricks documentation.